# 新数据变化时，怎样发现模型可能失效？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{
  "reference_accuracy": 0.9066666666666666,
  "window_results": {
    "正常": {
      "valid_rows": 400,
      "invalid_rows": 0,
      "ks_statistics": [
        0.035,
        0.0465
      ],
      "input_drift": false,
      "accuracy": 0.9175,
      "performance_alarm": false
    },
    "输入偏移": {
      "valid_rows": 400,
      "invalid_rows": 0,
      "ks_statistics": [
        0.572,
        0.05
      ],
      "input_drift": true,
      "accuracy": 0.9375,
      "performance_alarm": false
    },
    "概念改变": {
      "valid_rows": 400,
      "invalid_rows": 0,
      "ks_statistics": [
        0.0565,
        0.0455
      ],
      "input_drift": false,
      "accuracy": 0.13,
      "performance_alarm": true
    }
  },
  "invalid_rows_detected": 2,
  "thresholds": "teaching assumptions; calibrate for real deployment",
  "elapsed_seconds": 0.847
}


## 1. 补全：无标签口径
隐藏正常窗口标签，检查准确率未知而输入统计仍可用。

In [2]:
r = monitor(windows['正常'][0])
assert r['accuracy'] is None and len(r['ks_statistics']) == 2
print(r)

{'valid_rows': 400, 'invalid_rows': 0, 'ks_statistics': [0.035, 0.0465], 'input_drift': False, 'accuracy': None, 'performance_alarm': False}


## 2. 修改：错误数据结构
一列数据必须报错，坏记录必须检出。

In [3]:
try:
    monitor(np.zeros((10, 1)))
except ValueError:
    pass
else:
    raise AssertionError('schema错误未拒绝')
assert monitor(broken)['invalid_rows'] == 2
print('通过')

通过


## 3. 迁移：同输入不同关系
保持完全相同的X，反转标签；输入统计相同，两种准确率之和为1。

In [4]:
xx, yy = windows['正常']
a = monitor(xx, yy)
b = monitor(xx, 1 - yy)
assert a['ks_statistics'] == b['ks_statistics']
assert np.isclose(a['accuracy'] + b['accuracy'], 1)
print(a['accuracy'], b['accuracy'])

0.9175 0.0825
